In [1]:
import torch
from torch.utils.data import TensorDataset, ConcatDataset
from torch import Tensor
from data_utils import X, U, TIMES, extract_TensorDataset, extract_boundary
from plot_utils import plot_loss
from load_store_utils import resume_model
from typing import List
import os
import matplotlib.pyplot as plt
import numpy as np
import pickle

from data_utils import extract_boundary, extract_interior
from generate import X, U, DU, D2U, OUTWARD_NORMAL, PDE_VALUES

In [2]:
PHY_SYS = "Diffusion" # Stationary Allen-Cahn | Diffusion | Advection-Diffusion | Reaction-Diffusion
TRAVELLED_DIM = "" # Space | Parameters

In [3]:
if PHY_SYS == "Stationary Allen-Cahn":
    INTRA_TEST = "AllenCahn/data/intra_test_datasets.pth"
    INTER_TEST = "AllenCahn/data/inter_test_datasets.pth"
    MODEL_OFFLINE = "AllenCahn/FullDomainLearning/MultiTask/PINN_Norm1_GradClip_200/models2/trial0/model.pth"
    if TRAVELLED_DIM == "Space":
        DIR = "AllenCahn/DomainIncrementalLearning/MultiTask/PINN_CosLrAnnealing_Norm1_GradClip_LocalFullBd"
        MODEL0 = [f"{DIR}/Dom0_200/models2/trial0/model.pth"]
        FORGET_MODELS = [f"{DIR}/Forget/FineTune200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        REPLAY_MODELS = [f"{DIR}/LimitedMemory/Replay/Residual+Boundary/FineTuneLongMemory200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]]

        DISTILLED_DERS = [
            "Output", "Derivative", "Hessian", 
            "Output+Derivative", 
            "Output+Derivative+Hessian"
        ]
        DISTIL_ROW_IDS = [
            r"\textbf{Distil}$^{\mathbf{0}}$",
            r"\textbf{Distil}$^{\mathbf{1}}$",
            r"\textbf{Distil}$^{\mathbf{2}}$",
            r"\textbf{Distil}$^{\mathbf{0,1}}$",
            r"\textbf{Distil}$^{\mathbf{0,1,2}}$"
        ]
        
        DISTIL_MODELS = {}
        for der in DISTILLED_DERS:
            DISTIL_MODELS[der] = [f"{DIR}/LimitedMemory/Distill/{der}/FineTuneLongMemory200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        #EWC_MODELS = [f"{DIR}/LimitedMemory/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        EWC_MODELS = {
            "RunningAvg": [f"{DIR}/LimitedMemory/EWC/alpha_0.5/weight_auto/warmup1_decay1.0/FineTuneShortMemory200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]],
            "SparseMemory": [f"{DIR}/LimitedMemory/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory200/Dom{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        }
    elif TRAVELLED_DIM == "Parameters":
        DIR = "AllenCahn/TaskIncrementalLearning/PINN_CosLrAnnealing_Norm1_GradClip"
        MODEL0 = [f"{DIR}/Task0_200/models2/trial0/model.pth"]
        FORGET_MODELS = [f"{DIR}/Forget/FineTune200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        REPLAY_MODELS = [f"{DIR}/LimitedMemory/Replay/Residual+Boundary/FineTuneLongMemory200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        DISTILLED_DERS = ["Output", "Derivative", "Hessian", "Output+Derivative", "Output+Derivative+Hessian"]

        DISTIL_ROW_IDS = [
            r"\textbf{Distil}$^{\mathbf{0}}$",
            r"\textbf{Distil}$^{\mathbf{1}}$",
            r"\textbf{Distil}$^{\mathbf{2}}$",
            r"\textbf{Distil}$^{\mathbf{0,1}}$",
            r"\textbf{Distil}$^{\mathbf{0,1,2}}$"
        ]
        
        DISTIL_MODELS = {}
        for der in DISTILLED_DERS:
            DISTIL_MODELS[der] = [f"{DIR}/LimitedMemory/Distill/{der}/FineTuneLongMemory200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        #EWC_MODELS = [f"{DIR}/LimitedMemory/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]] 
        EWC_MODELS = {
            "RunningAvg": [f"{DIR}/LimitedMemory/EWC/alpha_0.5/weight_auto/warmup1_decay1.0/FineTuneShortMemory200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]],
            "SparseMemory": [f"{DIR}/LimitedMemory/EWC/alpha_1.0/weight_auto/warmup1_decay1.0/FineTuneLongMemory200/Task{i}/models/trial0/model.pth" for i in [1, 2, 3]]
        }
    else:
        raise ValueError(f"Unrecognized travelled dimention {TRAVELLED_DIM} for physical system {PHY_SYS}.")
    STATS_DIR = f"AllenCahn"
else: 
    TRAVELLED_DIM = ""
    if PHY_SYS == "Diffusion":
        INTRA_TEST = "AdvectionReactionDiffusion/Heat/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/intra_test_indices.pth"
        INTER_TEST = "AdvectionReactionDiffusion/Heat/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/intra_test_indices.pth"
        FULL_DATASET_INTRA = "AdvectionReactionDiffusion/Heat/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/full_datasets.pth"
        FULL_DATASET_INTER = "AdvectionReactionDiffusion/Heat/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/full_datasets.pth"
        MODEL_OFFLINE = {}
        for j in range(5):
            MODEL_OFFLINE[f"IC{j}"] = f"AdvectionReactionDiffusion/Heat/FullDomainLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration/ICD{j}/No_params_in_input/rectangle/PINN_NeumannBC_CosAnn_Std_GradClip_400/models2/trial0/model.pth"
        DIR = "AdvectionReactionDiffusion/Heat/TimeIncrementalLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration"
        STATS_DIR = "AdvectionReactionDiffusion/Heat"
    elif PHY_SYS == "Advection-Diffusion":
        INTRA_TEST = "AdvectionReactionDiffusion/ConvectionSlow/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/intra_test_indices.pth"
        INTER_TEST = "AdvectionReactionDiffusion/ConvectionSlow/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/intra_test_indices.pth"
        FULL_DATASET_INTRA = "AdvectionReactionDiffusion/ConvectionSlow/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/full_datasets.pth"
        FULL_DATASET_INTER = "AdvectionReactionDiffusion/ConvectionSlow/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/full_datasets.pth"
        MODEL_OFFLINE = {}
        for j in range(5):
            MODEL_OFFLINE[f"IC{j}"] = f"AdvectionReactionDiffusion/ConvectionSlow/FullDomainLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration/ICD{j}/No_params_in_input/rectangle/PINN_NeumannBC_CosAnn_Std_GradClip_400/models2/trial0/model.pth"
        DIR = "AdvectionReactionDiffusion/ConvectionSlow/TimeIncrementalLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration"
        STATS_DIR = "AdvectionReactionDiffusion/ConvectionSlow"
    elif PHY_SYS == "Reaction-Diffusion":
        INTRA_TEST = "AdvectionReactionDiffusion/AllenCahn/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/intra_test_indices.pth"
        INTER_TEST = "AdvectionReactionDiffusion/AllenCahn/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/intra_test_indices.pth"
        FULL_DATASET_INTRA = "AdvectionReactionDiffusion/AllenCahn/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/full_datasets.pth"
        FULL_DATASET_INTER = "AdvectionReactionDiffusion/AllenCahn/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_10-19/rep0/full_datasets.pth"
        MODEL_OFFLINE = {}
        for j in range(5):
            MODEL_OFFLINE[f"IC{j}"] = f"AdvectionReactionDiffusion/AllenCahn/FullDomainLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration/ICD{j}/No_params_in_input/rectangle/PINN_NeumannBC_CosAnn_Std_GradClip_400/models2/trial0/model.pth"
        DIR = "AdvectionReactionDiffusion/AllenCahn/TimeIncrementalLearning/5sources_ConstTimeStep_DT1_NeumannBcGeneration"
        STATS_DIR = "AdvectionReactionDiffusion/AllenCahn"
    else:
        raise ValueError(f"Unrecognized physical system {PHY_SYS}")
    MODEL0 = {}
    FORGET_MODELS = {}
    EWC_MODELS = {}
    REPLAY_MODELS = {}
    DISTIL_MODELS = {}

    DISTILLED_DERS = [
        "Output",
        "Derivative", "Derivative_x+Derivative_t",
        "Output+Derivative", "Output+Derivative_x+Derivative_t",
        "Output+Derivative+Hessian", "Output+Derivative_x+Derivative_t+Hessian_x+Hessian_t"
    ]
    DISTIL_ROW_IDS = [
        r"\textbf{Distil}$^{\mathbf{0}}$",
        r"\textbf{Distil}$^{\mathbf{1}}$",
        r"\textbf{Distil}$^{\mathbf{1}}_{\mathbf{x,t}}$",
        r"\textbf{Distil}$^{\mathbf{0,1}}$",
        r"\textbf{Distil}$^{\mathbf{0,1}}_{\mathbf{x,t}}$",
        r"\textbf{Distil}$^{\mathbf{0,1,2}}$",
        r"\textbf{Distil}$^{\mathbf{0,1,2}}_{\mathbf{x,t}}$"
    ]
    
    for j in range(5):
        DISTIL_MODELS[f"IC{j}"] = {}
        MODEL0[f"IC{j}"] = [f"{DIR}/ICD{j}/No_params_in_input/rectangle/T0/PINN_NeumannGlobalBC_CosAnn_Std_GradClip_200/models2/trial0/model.pth"]
        FORGET_MODELS[f"IC{j}"] = [f"{DIR}/ICD{j}/No_params_in_input/rectangle/Forget2/FineTune200/T{i}/PINN_NeumannBC_SeparatedIC_CosAnn_Std_GradClip_200/models/trial0/model.pth" for i in [1, 2, 3, 4]]
        EWC_MODELS[f"IC{j}"] = [f"{DIR}/ICD{j}/No_params_in_input/rectangle/EWC2/alpha_1.0/weight_auto/warmup1_decay1/FineTuneLongMemory200/T{i}/PINN_NeumannBC_SeparatedIC_CosAnn_Std_GradClip_200/models/trial0/model.pth" for i in [1, 2, 3, 4]]
        REPLAY_MODELS[f"IC{j}"] = [f"{DIR}/ICD{j}/No_params_in_input/rectangle/Replay2/Residual+Boundary/FineTuneLongMemory200/T{i}/PINN_NeumannBC_SeparatedIC_CosAnn_Std_GradClip_200/models/trial0/model.pth" for i in [1, 2, 3, 4]]
        for der in DISTILLED_DERS:
            DISTIL_MODELS[f"IC{j}"][der] = [f"{DIR}/ICD{j}/No_params_in_input/rectangle/Distill2/{der}/FineTuneLongMemory200/T{i}/PINN_NeumannBC_SeparatedIC_CosAnn_Std_GradClip_200/models/trial0/model.pth" for i in [1, 2, 3, 4]]


In [4]:
COL_IDS = [
    r"$\mathcal{L}[u^{\theta^*}, u]$",
    r"$\mathcal{L}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$",
    r"$\mathcal{L}[\nabla_x u^{\theta^*}, \nabla_x u]$",
    r"$\mathcal{L}[\nabla_t u^{\theta^*}, \nabla_t u]$",
    r"$\mathcal{L}[\nabla^2_{x,t} u^{\theta^*}, \nabla^2_{x,t} u]$",
    r"$\mathcal{L}[\mathcal{G}[u^{\theta^*}], \mathcal{G}[u]]$"
    ]
ROW_IDS = [
    r"\textbf{Joint}",
    r"\textbf{Forget}",
    r"\textbf{EWC$_{\alpha}$}",
    r"\textbf{EWC}",
    r"\textbf{Replay}",
    r"\textbf{Distil$^\mathbf{0}$}}",
    r"\textbf{Distil$^\mathbf{1}$}}",
    r"\textbf{Distil$^\mathbf{0,1}$}}",
    r"\textbf{Distil$^\mathbf{0,1,2}$}}",
    r"\textbf{Distil$^\mathbf{1}_\mathbf{x, t}$}}",
    r"\textbf{Distil$^\mathbf{0,1}_\mathbf{x, t}$}}",
    r"\textbf{Distil$^\mathbf{0,1,2}_\mathbf{x, t}$}}"
]

In [5]:
COLOR_SCALES = {
    "orange": [r"\cellcolor[HTML]{FFBB56}", r"\cellcolor[HTML]{FFD981}"],
    "blue": [r"\cellcolor[HTML]{B2C3FD}", r"\cellcolor[HTML]{D4DEFF}"],
    "gray": [r"\cellcolor[HTML]{E6DCDC}"],
    "red": [r"\cellcolor[HTML]{EDCACA}", r"\cellcolor[HTML]{F8ECEC}"]
}
COL_ID_COLOR = {
    "orange": r"\color[HTML]{A34900}",
    "blue": r"\color[HTML]{4D5AD0}",
    "red": r"\color[HTML]{9C3535}"
}
ROW_ID_COLOR = {
    "orange": r"\color[HTML]{D15E00}",
    "blue": r"\color[HTML]{687FFF}",
    "red": r"\color[HTML]{BE4141}"
}

HLINE_COLOR = {
    "orange": r"\arrayrulecolor[HTML]{D15E00}",
    "blue": r"\arrayrulecolor[HTML]{687FFF}"
}

In [6]:
def include_time_in_input(dataset: TensorDataset) -> TensorDataset:
    """
    Includes the temporal coordinates in the input column.

    Parameters
    ----------
    dataset : TensorDataset

    Returns
    -------
    TensorDataset
        The modified TensorDataset.
    """
    if dataset is None:
        return None
    cols = list(dataset.tensors)
    cols[0] = torch.cat([cols[X], cols[TIMES].unsqueeze(1)], dim=1)
    return TensorDataset(*cols)

def subsample(
        datasets: List[TensorDataset],
        indicess: Tensor|List[Tensor]
    ) -> List[TensorDataset]:
    subsampless = [[] for ic in range(len(indicess))]
    for ic in range(len(indicess)):
        trajectory = datasets[ic].datasets
        trajectory_indices = indicess[ic]
        for instant, indices in zip(trajectory, trajectory_indices):
            cols = [col[indices] for col in instant.tensors]
            subsampless[ic].append(TensorDataset(*cols))
    return subsampless

def load_ds(dataset_path: str, indices_path: str = None, incremental_dim = "Time"):
    if not os.path.exists(dataset_path):
        raise ValueError(f"'File {dataset_path}' not found.")
    full_dataset = torch.load(os.path.join(dataset_path), weights_only=False)

    data_dict = {}

    if incremental_dim == "Time":
        if not os.path.exists(indices_path):
            raise ValueError(f"'File {indices_path}' not found.")
        indicess = torch.load(os.path.join(indices_path))

        datasets = subsample(full_dataset.datasets, indicess)
        del full_dataset

        for i, ds in enumerate(datasets):
            data_dict[f"IC{i}"] = {}       
            data_dict[f"IC{i}"]["all_tasks"] = include_time_in_input(merge_ds(ds))
            data_dict[f"IC{i}"]["all_tasks_no_ic"] = include_time_in_input(merge_ds(ds[1:]))
            for j, task in enumerate([[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]):
                data_dict[f"IC{i}"][f"T{j}"] = include_time_in_input(merge_ds(datasets=[ds[t] for t in task]))              
            data_dict[f"IC{i}"]["ic"] = include_time_in_input(ds[0])
    elif incremental_dim == "Space":
        datasets = [ds.datasets[0] for ds in full_dataset.datasets]
        print(len(datasets))
        dataset = merge_ds(datasets)
        spatial_ranges = [
            {"x": [0.0, 1.0], "y": [0.0, 1.0]},
            {"x": [0.0, 1.0], "y": [-1.0, 0.0]},
            {"x": [-1.0, 0.0], "y": [-1.0, 0.0]},
            {"x": [-1.0, 0.0], "y": [0.0, 1.0]}
        ]
        tasks = [extract_TensorDataset(dataset=dataset, spatial_ranges=s) for s in spatial_ranges]
        data_dict["all_tasks"] = dataset
        for j, task in enumerate(tasks):
            data_dict[f"T{j}"] = task
    elif incremental_dim == "Parameters":
        datasets = [ds.datasets[0] for ds in full_dataset.datasets]
        print(len(datasets))
        dataset = merge_ds(datasets)
        tasks = datasets
        data_dict["all_tasks"] = dataset
        for j, task in enumerate(tasks):
            data_dict[f"T{j}"] = task
    else:
        raise ValueError(f"Unrecognized incremental dimention '{incremental_dim}'.")
    return data_dict

def merge_ds(datasets: List[TensorDataset]) -> TensorDataset:
    cols = None   
    for ds in datasets:
        new_cols = list(ds.tensors)
        if cols is None:
            cols = new_cols
        else:
            for i, col in enumerate(new_cols):
                cols[i] = torch.cat([cols[i], col])
    return TensorDataset(*cols)

def prepare_dataset(datasets: List[List[TensorDataset]], time_instants: List[int] = None) -> List:
    '''
    Return [TensorDataset for t in time_instants]
    '''
    if time_instants is None:
        n_snapshots = len(datasets[0])
        time_instants = [i for i in range(n_snapshots)]
    data = []
    for i in time_instants:
        data.append(merge_ds([trajectory[i] for trajectory in datasets]))
    return data

def evaluate(
        model: str,
        dataset: TensorDataset,
        ic_dataset: TensorDataset = None,
        split_space_time: bool = False,
        pde_params_in_input: bool = False,
        bcs: str = "Neumann"
    ) -> tuple:
    """
    Evaluate a model on a dataset and return loss terms values.

    Parameters
    ----------
    model_path : str
        Model file path.
    dataset : str|TensorDataset
        Dataset on which the model is evaluated.
    split_space_time : bool

    Returns
    -------
    tuple
        out_loss, der_loss, (der_loss_x, der_loss_t,) res_loss, bc_loss(, ic_loss)
    """
    model = resume_model(model_path=model)
    if split_space_time:
        out, der, derx, dert, hes, hesx, hest, res = list(model.evaluate(dataset=dataset, split_space_time=split_space_time))
        losses = [out, der, derx, dert, res]
    else:
        out, der, hes, res = list(model.evaluate(dataset=dataset, split_space_time=split_space_time))
        losses = [out, der, res]

    interior = extract_interior(dataset)
    losses_int = model.evaluate(dataset=interior, split_space_time=split_space_time)
    losses[-1] = losses_int[-1]

    boundary = extract_boundary(dataset)
    x = boundary.tensors[X].to(torch.float32)
    if pde_params_in_input:
        pde_params = boundary.tensors[PDE_VALUES].to(torch.float32)
    else:
        pde_params = None
    du = boundary.tensors[DU]
    n = boundary.tensors[OUTWARD_NORMAL]
    out_flux = (du[:, :2] * n).sum(dim=1).reshape(-1)
    model.eval()
    with torch.no_grad():
        if bcs == "Neumann":
            du_pred = model.derivative(1, x, pde_params).to(torch.float64)
            out_flux_pred = (du_pred[:, :2] * n).sum(dim=1).reshape(-1)
            bc_loss = model.loss_container(out_flux_pred, out_flux)    
            losses.append(bc_loss)
        elif bcs == "Dirichlet":
            if split_space_time:
                bc_out, _, _, _, _, _, _, _ = list(model.evaluate(dataset=boundary, split_space_time=split_space_time))
                losses.append(bc_out)
            else:
                bc_out, _, _, _ = list(model.evaluate(dataset=boundary, split_space_time=split_space_time))
                losses.append(bc_out)
        else:
            raise ValueError(f"Unknown {bcs} BCs.")

        if ic_dataset is not None:
            x = ic_dataset.tensors[X].to(torch.float32)
            u = ic_dataset.tensors[U]
            u_pred = model.forward(x).to(torch.float64)
            losses.append(model.loss_container(u_pred, u))

    del dataset
    return losses

def compute_avg(losses: list) -> float:
    return np.mean(losses)

def compute_stddev(losses: list) -> float:
    return np.std(losses)

def get_colors(values: List[float], color_scale, f: str = "min") -> List[str]:
    '''
        Return the color string for each item in the values list.
    '''
    # create a sorted version of the unique elements
    # using set() handles duplicates by giving them the same rank
    sorted_unique = sorted(list(set(values)))
    if f == "max":
        sorted_unique.reverse()
    
    # create a lookup dictionary: {value: rank}
    rank_map = {val: i for i, val in enumerate(sorted_unique)}
    
    # rank for each item in the original list
    ranks = [rank_map[x] for x in values]
    colors = []
    for rank in ranks:
        if rank < len(color_scale):
            color = color_scale[rank]
            colors.append(color)
        else:
            colors.append("")
    return colors

def latex_sci(value, ub = None):
    v = value
    value = "{:.2e}".format(value)
    base, exponent = value.split("e")
    # remove leading zeros and plus signs from exponent
    exponent = int(exponent)
    if ub is not None and v < ub:
        return r"$\mathbf{" + str(base) + r" \times 10^{" + str(exponent) + r"}}$"
    return r"$" + str(base) + r" \times 10^{" + str(exponent) + r"}$"

def latex_table(
        columns: List[float], 
        col_ids: List[str], 
        row_ids: List[str], 
        colorss: List[List[str]] = None,
        col_id_color: str = "", 
        row_id_color: str = "", 
        hline_color: str = None, 
        diag_ids: bool = False,
        ub: bool = True,
        caption: str = ""
    ) -> str:
    if diag_ids:
        dcol = r"\dcol"
        drow = r"\drow"
    else:
        dcol = r""
        drow = r""
    table = r"\begin{table}[H]" + "\n" + \
        r"\centering" + "\n" + \
        r"\caption{" + caption + r"}" + "\n" + \
        r"\label{}" + "\n" + \
        r"\begin{tabular}{lccccc}" + "\n" + \
        r"\toprule" + "\n"
    for col_id in col_ids:
        table += (r" & " + col_id_color + dcol + r"{" + col_id + r"}")
    table += (r" \\" + "\n" + r"\midrule" + "\n")
    for i, row_id in enumerate(row_ids):
        table += (row_id_color + drow + r"{" + row_id + r"}")
        if colorss != None:
            if ub:
                for column, colors in zip(columns, colorss):
                    table += (r" & " + colors[i] + latex_sci(value=column[i], ub=column[0]))
            else:
                for column, colors in zip(columns, colorss):
                    table += (r" & " + colors[i] + latex_sci(value=column[i]))
        else:
            for column in columns:
                if ub:
                    table += (r" & " + latex_sci(value=column[i], ub=column[0]))
                else:
                    table += (r" & " + latex_sci(value=column[i]))
        table += (r" \\" + "\n")
        if i == 0 and hline_color is not None:
            table += (hline_color + r"\midrule\arrayrulecolor[HTML]{000000}" + "\n")
    table += (r"\bottomrule" + "\n" + r"\end{tabular}" + "\n" + r"\end{table}" + "\n")
    return table

def per_model_per_task_loss(models: List[str], tasks: List[TensorDataset], ic_task: TensorDataset = None, der_tasks: List[TensorDataset] = None, split_space_time: bool = False, pde_params_in_input: bool = False, bcs: str = "Neumann"):
    loss_dict = {}
    for k, model in enumerate(models):
        loss_dict[f"model{k}"] = {}
        for j, task in enumerate(tasks):
            loss_dict[f"model{k}"][f"task{j}"] = {}
            losses = evaluate(model=model, dataset=task, ic_dataset=ic_task, split_space_time=split_space_time, pde_params_in_input=pde_params_in_input, bcs=bcs)
            if der_tasks != None and (j != 0 or len(tasks) == 1):
                losses2 = evaluate(model=model, dataset=der_tasks[j], ic_dataset=ic_task, split_space_time=split_space_time, pde_params_in_input=pde_params_in_input, bcs=bcs)
            else:
                losses2 = losses
            loss_dict[f"model{k}"][f"task{j}"]["out"] = losses[0]
            loss_dict[f"model{k}"][f"task{j}"]["der"] = losses2[1]
            i = 2
            if split_space_time:
                loss_dict[f"model{k}"][f"task{j}"]["derx"] = losses2[2]
                loss_dict[f"model{k}"][f"task{j}"]["dert"] = losses2[3]
                i = 4
            loss_dict[f"model{k}"][f"task{j}"]["res"] = losses[i]
            loss_dict[f"model{k}"][f"task{j}"]["bc"] = losses[i+1]
            if ic_task is not None:
                loss_dict[f"model{k}"][f"task{j}"]["ic"] = losses[i+2]    
    return loss_dict

def forgetting(loss_dict, loss_id):
    forgetting = {}
    for model in loss_dict.keys():
        if model != "model0":
            forgetting[model] = {}
            m = int(model[-1])
            for task in loss_dict[model].keys():
                t = int(task[-1])
                if t >= m:
                    break
                else:
                    loss_m_t = loss_dict[model][task][loss_id]
                    min_loss_t = min([loss_dict[f"model{i}"][task][loss_id] for i in range(m)])
                    forgetting[model][task] = loss_m_t - min_loss_t
            forgetting[model]["avg"] = np.mean([forgetting[model][task] for task in forgetting[model].keys()])
    models = list(forgetting.keys())
    forgetting["avg"] = np.mean([forgetting[model]["avg"] for model in models])
    forgetting["stddev"] = np.std([forgetting[model]["avg"] for model in models])
    return forgetting

def backward_transfer(loss_dict, loss_id):
    back_transfer = {}
    for model in loss_dict.keys():
        if model != "model0":
            m = int(model[-1])
            back_transfer[model] = np.mean([loss_dict[f"model{i}"][f"task{i}"][loss_id] - loss_dict[model][f"task{i}"][loss_id] for i in range(m)])
    models = list(back_transfer.keys())
    back_transfer["avg"] = np.mean([back_transfer[model] for model in models])
    back_transfer["stddev"] = np.std([back_transfer[model] for model in models])
    return back_transfer

def incremental_loss(loss_dict, loss_id):
    incremental_loss = {}
    for model in loss_dict.keys():
        if model != "model0":
            m = int(model[-1])
            incremental_loss[model] = sum(loss_dict[model][f"task{t}"][loss_id] for t in range(m+1))
    models = list(incremental_loss.keys())
    incremental_loss["avg"] = np.mean([incremental_loss[model] for model in models])
    incremental_loss["stddev"] = np.std([incremental_loss[model] for model in models])
    return incremental_loss


In [7]:
if PHY_SYS != "Stationary Allen-Cahn":
    for filename, indices_path, full_dataset in [("intra_test_stats", INTRA_TEST, FULL_DATASET_INTRA), ("inter_test_stats", INTER_TEST, FULL_DATASET_INTER)]:
        print(f"Computing {filename} ...")
        stats = {
            "forget": {},
            "ewc": {},
            "replay": {}
        }
        for der in DISTILLED_DERS:
            stats[f"distil_{der}"] = {}

        data_dict = load_ds(dataset_path=full_dataset, indices_path=indices_path, incremental_dim="Time")
        for mode in stats.keys():
            stats[mode]["losses"] = {
                "all_tasks": {},
                "per_task": {}
            }
            stats[mode]["forgetting"] = {}
            stats[mode]["backward_transfer"] = {}
            stats[mode]["incremental_loss"] = {}
            for i in range(5):
                if mode == "replay":
                    models = MODEL0[f"IC{i}"] + REPLAY_MODELS[f"IC{i}"]
                elif mode == "forget":
                    models = MODEL0[f"IC{i}"] + FORGET_MODELS[f"IC{i}"]
                elif mode == "ewc":
                    models = MODEL0[f"IC{i}"] + EWC_MODELS[f"IC{i}"]
                else:
                    der = mode.split(sep="distil_")[1]
                    models = MODEL0[f"IC{i}"] + DISTIL_MODELS[f"IC{i}"][der]

                if filename == "intra_test_stats":
                    loss_dict_final_model_all_tasks = per_model_per_task_loss(models=[models[-1]], tasks=[data_dict[f"IC{i}"]["all_tasks"]], ic_task=data_dict[f"IC{i}"]["ic"], der_tasks=[data_dict[f"IC{i}"]["all_tasks_no_ic"]], split_space_time=True)
                else:
                    loss_dict_final_model_all_tasks = per_model_per_task_loss(models=[models[-1]], tasks=[data_dict[f"IC{i}"]["all_tasks"]], ic_task=data_dict[f"IC{i}"]["ic"], split_space_time=True)

                loss_dict_per_model_per_task = per_model_per_task_loss(models=models, tasks=[data_dict[f"IC{i}"][f"T{j}"] for j in range(5)], ic_task=data_dict[f"IC{i}"]["ic"], split_space_time=True)

                forgetting_dict = forgetting(loss_dict=loss_dict_per_model_per_task, loss_id="out")

                backward_transfer_dict = backward_transfer(loss_dict=loss_dict_per_model_per_task, loss_id="out")

                incremental_loss_dict = incremental_loss(loss_dict=loss_dict_per_model_per_task, loss_id="out")

                stats[mode]["losses"]["all_tasks"][f"IC{i}"] = loss_dict_final_model_all_tasks
                stats[mode]["losses"]["per_task"][f"IC{i}"] = loss_dict_per_model_per_task
                stats[mode]["forgetting"][f"IC{i}"] = forgetting_dict
                stats[mode]["backward_transfer"][f"IC{i}"] = backward_transfer_dict
                stats[mode]["incremental_loss"][f"IC{i}"] = incremental_loss_dict

            for k in ["out", "der", "derx", "dert", "res", "bc", "ic"]:
                stats[mode]["losses"]["all_tasks"][f"avg_{k}"] = np.mean([stats[mode]["losses"]["all_tasks"][f"IC{i}"]["model0"]["task0"][k] for i in range(5)])
                stats[mode]["losses"]["all_tasks"][f"stddev_{k}"] = np.std([stats[mode]["losses"]["all_tasks"][f"IC{i}"]["model0"]["task0"][k] for i in range(5)])

            stats[mode]["forgetting"]["avg"] = np.mean([stats[mode]["forgetting"][f"IC{i}"]["avg"] for i in range(5)])
            stats[mode]["forgetting"]["stddev"] = np.std([stats[mode]["forgetting"][f"IC{i}"]["avg"] for i in range(5)])

            stats[mode]["backward_transfer"]["avg"] = np.mean([stats[mode]["backward_transfer"][f"IC{i}"]["avg"] for i in range(5)])
            stats[mode]["backward_transfer"]["stddev"] = np.std([stats[mode]["backward_transfer"][f"IC{i}"]["avg"] for i in range(5)])

            stats[mode]["incremental_loss"]["avg"] = np.mean([stats[mode]["incremental_loss"][f"IC{i}"]["avg"] for i in range(5)])
            stats[mode]["incremental_loss"]["stddev"] = np.std([stats[mode]["incremental_loss"][f"IC{i}"]["avg"] for i in range(5)])

        stats["offline"] = {}
        for i in range(5):
            if filename == "intra_test_stats":
                stats["offline"][f"IC{i}"] = per_model_per_task_loss(models=[MODEL_OFFLINE[f"IC{i}"]], tasks=[data_dict[f"IC{i}"]["all_tasks"]], ic_task=data_dict[f"IC{i}"]["ic"], der_tasks=[data_dict[f"IC{i}"]["all_tasks_no_ic"]], split_space_time=True)
            else:
                stats["offline"][f"IC{i}"] = per_model_per_task_loss(models=[MODEL_OFFLINE[f"IC{i}"]], tasks=[data_dict[f"IC{i}"]["all_tasks"]], ic_task=data_dict[f"IC{i}"]["ic"], split_space_time=True)

        for k in ["out", "der", "derx", "dert", "res", "bc", "ic"]:
            stats["offline"][f"avg_{k}"] = np.mean([stats["offline"][f"IC{i}"]["model0"]["task0"][k] for i in range(5)])
            stats["offline"][f"stddev_{k}"] = np.std([stats["offline"][f"IC{i}"]["model0"]["task0"][k] for i in range(5)])
        
        with open(f"{STATS_DIR}/{filename}.pkl", "wb") as f:
            pickle.dump(stats, f)
        del data_dict
else:
    for filename, full_dataset in [("intra_test_stats", INTRA_TEST), ("inter_test_stats", INTER_TEST)]:
        stats = {
            "forget": {},
            "ewc": {},
            "ewc_alpha": {},
            "replay": {}
        }
        for der in DISTILLED_DERS:
            stats[f"distil_{der}"] = {}

        data_dict = load_ds(dataset_path=full_dataset, incremental_dim=TRAVELLED_DIM)

        for mode in stats.keys():
            stats[mode]["losses"] = {
                "all_tasks": {},
                "per_task": {}
            }
            stats[mode]["forgetting"] = {}
            stats[mode]["backward_transfer"] = {}
            stats[mode]["incremental_loss"] = {}
            if mode == "replay":
                models = MODEL0 + REPLAY_MODELS
            elif mode == "forget":
                models = MODEL0 + FORGET_MODELS
            elif mode == "ewc":
                models = MODEL0 + EWC_MODELS["SparseMemory"]
            elif mode == "ewc_alpha":
                models = MODEL0 + EWC_MODELS["RunningAvg"]
            else:
                der = mode.split(sep="_")[1]
                models = MODEL0 + DISTIL_MODELS[der]
            loss_dict_final_model_all_tasks = per_model_per_task_loss(models=[models[-1]], tasks=[data_dict["all_tasks"]], pde_params_in_input=True, bcs="Dirichlet")

            loss_dict_per_model_per_task = per_model_per_task_loss(models=models, tasks=[data_dict[f"T{j}"] for j in range(4)], pde_params_in_input=True, bcs="Dirichlet")

            forgetting_dict = forgetting(loss_dict=loss_dict_per_model_per_task, loss_id="out")

            backward_transfer_dict = backward_transfer(loss_dict=loss_dict_per_model_per_task, loss_id="out")

            incremental_loss_dict = incremental_loss(loss_dict=loss_dict_per_model_per_task, loss_id="out")

            stats[mode]["losses"]["all_tasks"] = loss_dict_final_model_all_tasks
            stats[mode]["losses"]["per_task"] = loss_dict_per_model_per_task
            stats[mode]["forgetting"] = forgetting_dict
            stats[mode]["backward_transfer"] = backward_transfer_dict
            stats[mode]["incremental_loss"] = incremental_loss_dict

        stats["offline"] = {}
        stats["offline"] = per_model_per_task_loss(models=[MODEL_OFFLINE], tasks=[data_dict["all_tasks"]], pde_params_in_input=True, bcs="Dirichlet")
        
        with open(f"{STATS_DIR}/{filename}_{TRAVELLED_DIM}.pkl", "wb") as f:
            pickle.dump(stats, f)
        del data_dict

Computing intra_test_stats ...
Computing inter_test_stats ...


In [8]:
if TRAVELLED_DIM != "":
    TRAVELLED_DIM = "_" + TRAVELLED_DIM
with open(f"{STATS_DIR}/intra_test_stats{TRAVELLED_DIM}.pkl", "rb") as f:
    intra_stats = pickle.load(f)
with open(f"{STATS_DIR}/inter_test_stats{TRAVELLED_DIM}.pkl", "rb") as f:
    inter_stats = pickle.load(f)

In [9]:
first = True
for stats, caption, color in [(intra_stats, "Internal test", "orange"), (inter_stats, "External test", "blue")]:
    columns = {}
    if PHY_SYS == "Stationary Allen-Cahn":
        for k in ["out", "der", "res", "bc"]:
            columns[k] = []
            columns[k].append(stats["offline"]["model0"]["task0"][k])
            for mode in ["forget", "ewc", "ewc_alpha", "replay"] + [f"distil_{der}" for der in DISTILLED_DERS]:
                columns[k].append(stats[mode]["losses"]["all_tasks"]["model0"]["task0"][k])
    else:
        for k in ["out", "der", "res", "bc", "ic"]:
            columns[k] = []
            columns[k].append(stats["offline"][f"avg_{k}"])
            for mode in ["forget", "replay"] + [f"distil_{der}" for der in DISTILLED_DERS]:
                columns[k].append(stats[mode]["losses"]["all_tasks"][f"avg_{k}"])

    if PHY_SYS == "Stationary Allen-Cahn":
        col_ids_sys = [
            r"$\mathcal{L}[\mathcal{G}[u^{\theta^*}], 0]$",
            r"$\mathcal{L}[\mathcal{B}[u^{\theta^*}], 0]$"
        ]
        cols_sys = [columns["res"], columns["bc"]]
    else:
        col_ids_sys = [
            r"$\mathcal{L}[\mathcal{G}[u^{\theta^*}], 0]$",
            r"$\mathcal{L}[\mathcal{B}[u^{\theta^*}], 0]$",
            r"$\mathcal{L}[\mathcal{I}[u^{\theta^*}], 0]$"
        ]
        cols_sys = [columns["res"], columns["bc"], columns["ic"]]
    if PHY_SYS == "Stationary Allen-Cahn":
        col_ids_ders = [
            r"$\mathcal{L}[u^{\theta^*}, u]$",
            r"$\mathcal{L}[\nabla_x u^{\theta^*}, \nabla_x u]$"
        ]
    else:
        col_ids_ders = [
            r"$\mathcal{L}[u^{\theta^*}, u]$",
            r"$\mathcal{L}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$"
        ]
    cols_ders = [columns["out"], columns["der"]]

    if PHY_SYS == "Stationary Allen-Cahn":
        row_ids = [
            r"\textbf{Joint}",
            r"\textbf{Forget}",
            r"\textbf{EWC}",
            r"$\mathbf{EWC}_{\alpha}$",
            r"\textbf{Replay}"
        ] + DISTIL_ROW_IDS
    else:
        row_ids = [
            r"\textbf{Joint}",
            r"\textbf{Forget}",
            r"\textbf{Replay}"
        ] + DISTIL_ROW_IDS

    for cols, col_ids in [(cols_sys, col_ids_sys), (cols_ders, col_ids_ders)]:
        table = latex_table(
            columns=cols, 
            col_ids=col_ids, 
            row_ids=row_ids, 
            diag_ids=False, 
            colorss=[[""]+get_colors(col[1:], COLOR_SCALES[color]) for col in cols], 
            col_id_color=COL_ID_COLOR[color], 
            row_id_color=ROW_ID_COLOR[color], 
            hline_color=HLINE_COLOR[color],
            caption=caption
        ) +"\n"
        if first:
            with open(f"{STATS_DIR}/loss_tables{TRAVELLED_DIM}.txt", "w") as file:
                file.write(table)
            first = False
        else:
            with open(f"{STATS_DIR}/loss_tables{TRAVELLED_DIM}.txt", "a") as file:
                file.write(table)
        print(table)

\begin{table}[H]
\centering
\caption{Internal test}
\label{}
\begin{tabular}{lccccc}
\toprule
 & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{G}[u^{\theta^*}], 0]$} & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{B}[u^{\theta^*}], 0]$} & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{I}[u^{\theta^*}], 0]$} \\
\midrule
\color[HTML]{D15E00}{\textbf{Joint}} & $1.59 \times 10^{-3}$ & $5.54 \times 10^{-4}$ & $1.06 \times 10^{-3}$ \\
\arrayrulecolor[HTML]{D15E00}\midrule\arrayrulecolor[HTML]{000000}
\color[HTML]{D15E00}{\textbf{Forget}} & $7.46 \times 10^{0}$ & $1.53 \times 10^{0}$ & $1.06 \times 10^{-2}$ \\
\color[HTML]{D15E00}{\textbf{Replay}} & \cellcolor[HTML]{FFBB56}$2.09 \times 10^{-2}$ & \cellcolor[HTML]{FFBB56}$1.06 \times 10^{-1}$ & $2.03 \times 10^{-2}$ \\
\color[HTML]{D15E00}{\textbf{Distil}$^{\mathbf{0}}$} & $5.62 \times 10^{-1}$ & $1.47 \times 10^{0}$ & $1.11 \times 10^{-2}$ \\
\color[HTML]{D15E00}{\textbf{Distil}$^{\mathbf{1}}$} & \cellcolor[HTML]{FFD981}$7.22 \times 10^{-2}$ & $2.48 \t

In [ ]:
columns = {}
for k in ["forgetting", "backward_transfer", "incremental_loss"]:
    columns[k] = []
    for mode in ["forget", "replay"] + [f"distil_{der}" for der in DISTILLED_DERS]:
        columns[k].append(intra_stats[mode][k][f"avg"])

col_ids = [
    r"Forgetting",
    r"Backward Transfer",
    r"Incremental Loss"
]
cols = [columns["forgetting"], columns["backward_transfer"], columns["incremental_loss"]]

row_ids = [
    r"\textbf{Forget}",
    #r"\textbf{EWC}",
    #r"$\mathbf{EWC}_{\alpha}$",
    r"\textbf{Replay}"
] + DISTIL_ROW_IDS

table = latex_table(
    columns=cols, 
    col_ids=col_ids, 
    row_ids=row_ids, 
    diag_ids=False, 
    ub=False, 
    colorss=[
        get_colors(cols[0], COLOR_SCALES["gray"]), 
        get_colors(cols[1], COLOR_SCALES["gray"], "max"), 
        get_colors(cols[2], COLOR_SCALES["gray"])
    ]
)

with open(f"{STATS_DIR}/CLstats_tables{TRAVELLED_DIM}.txt", "w") as file:
    file.write(table)
print(table)

KeyError: 'ewc_alpha'

# Std dev

In [ ]:
if TRAVELLED_DIM != "":
    TRAVELLED_DIM = "_" + TRAVELLED_DIM
with open(f"{STATS_DIR}/intra_test_stats{TRAVELLED_DIM}.pkl", "rb") as f:
    intra_stats = pickle.load(f)
with open(f"{STATS_DIR}/inter_test_stats{TRAVELLED_DIM}.pkl", "rb") as f:
    inter_stats = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'AllenCahn/intra_test_stats__Space.pkl'

In [ ]:
if PHY_SYS != "Stationary Allen-Cahn":
    first = True
    for stats, caption, color in [(intra_stats, "Internal test", "orange"), (inter_stats, "External test", "blue")]:
        columns = {}
        for k in ["out", "der", "res", "bc", "ic"]:
            columns[k] = []
            columns[k].append(stats["offline"][f"stddev_{k}"])
            for mode in ["forget", "ewc", "replay"] + [f"distil_{der}" for der in DISTILLED_DERS]:
                columns[k].append(stats[mode]["losses"]["all_tasks"][f"stddev_{k}"])

        col_ids_sys = [
            r"$\mathcal{L}[\mathcal{G}[u^{\theta^*}], 0]$",
            r"$\mathcal{L}[\mathcal{B}[u^{\theta^*}], 0]$",
            r"$\mathcal{L}[\mathcal{I}[u^{\theta^*}], 0]$"
        ]
        cols_sys = [columns["res"], columns["bc"], columns["ic"]]
        
        col_ids_ders = [
            r"$\mathcal{L}[u^{\theta^*}, u]$",
            r"$\mathcal{L}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$"
        ]
        cols_ders = [columns["out"], columns["der"]]

        row_ids = [
            r"\textbf{Joint}",
            r"\textbf{Forget}",
            r"\textbf{EWC}",
            r"\textbf{Replay}"
        ] + DISTIL_ROW_IDS

        for cols, col_ids in [(cols_sys, col_ids_sys), (cols_ders, col_ids_ders)]:
            table = latex_table(
                columns=cols, 
                col_ids=col_ids, 
                row_ids=row_ids, 
                diag_ids=False, 
                colorss=None, 
                col_id_color=COL_ID_COLOR[color], 
                row_id_color=ROW_ID_COLOR[color], 
                hline_color=None,
                caption=caption,
                ub=False
            ) +"\n"
            if first:
                with open(f"{STATS_DIR}/loss_tables{TRAVELLED_DIM}_stddev.txt", "w") as file:
                    file.write(table)
                first = False
            else:
                with open(f"{STATS_DIR}/loss_tables{TRAVELLED_DIM}_stddev.txt", "a") as file:
                    file.write(table)
            print(table)

\begin{table}[H]
\centering
\caption{Internal test}
\label{}
\begin{tabular}{lccccc}
\toprule
 & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{G}[u^{\theta^*}], 0]$} & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{B}[u^{\theta^*}], 0]$} & \color[HTML]{A34900}{$\mathcal{L}[\mathcal{I}[u^{\theta^*}], 0]$} \\
\midrule
\color[HTML]{D15E00}{\textbf{Joint}} & $1.05 \times 10^{-3}$ & $4.48 \times 10^{-4}$ & $7.11 \times 10^{-4}$ \\
\color[HTML]{D15E00}{\textbf{Forget}} & $4.78 \times 10^{0}$ & $8.24 \times 10^{-1}$ & $8.34 \times 10^{-3}$ \\
\color[HTML]{D15E00}{\textbf{EWC}} & $3.85 \times 10^{0}$ & $1.08 \times 10^{0}$ & $2.60 \times 10^{-2}$ \\
\color[HTML]{D15E00}{\textbf{Replay}} & $2.00 \times 10^{-2}$ & $1.20 \times 10^{-1}$ & $2.11 \times 10^{-2}$ \\
\color[HTML]{D15E00}{\textbf{Distil}$^{\mathbf{0}}$} & $3.51 \times 10^{-1}$ & $8.11 \times 10^{-1}$ & $7.76 \times 10^{-3}$ \\
\color[HTML]{D15E00}{\textbf{Distil}$^{\mathbf{1}}$} & $5.37 \times 10^{-2}$ & $2.03 \times 10^{-1}$ & $3.11 \times 

In [ ]:
columns = {}
for k in ["forgetting", "backward_transfer", "incremental_loss"]:
    columns[k] = []
    for mode in ["forget", "ewc", "replay"] + [f"distil_{der}" for der in DISTILLED_DERS]:
        columns[k].append(intra_stats[mode][k][f"stddev"])

col_ids = [
    r"Forgetting",
    r"Backward Transfer",
    r"Incremental Loss"
]
cols = [columns["forgetting"], columns["backward_transfer"], columns["incremental_loss"]]

row_ids = [
    r"\textbf{Forget}",
    r"\textbf{EWC}",
    r"\textbf{Replay}"
] + DISTIL_ROW_IDS

table = latex_table(
    columns=cols, 
    col_ids=col_ids, 
    row_ids=row_ids, 
    diag_ids=False, 
    ub=False, 
    colorss=None
)

with open(f"{STATS_DIR}/CLstats_tables{TRAVELLED_DIM}_stddev.txt", "w") as file:
    file.write(table)
print(table)

\begin{table}[H]
\centering
\caption{}
\label{}
\begin{tabular}{lccccc}
\toprule
 & {Forgetting} & {Backward Transfer} & {Incremental Loss} \\
\midrule
{\textbf{Forget}} & $8.77 \times 10^{-3}$ & $8.36 \times 10^{-3}$ & $5.74 \times 10^{-2}$ \\
{\textbf{EWC}} & $1.06 \times 10^{-2}$ & $1.05 \times 10^{-2}$ & $7.04 \times 10^{-2}$ \\
{\textbf{Replay}} & $2.26 \times 10^{-4}$ & $1.35 \times 10^{-4}$ & $2.43 \times 10^{-3}$ \\
{\textbf{Distil}$^{\mathbf{0}}$} & $2.56 \times 10^{-4}$ & $2.88 \times 10^{-4}$ & $1.41 \times 10^{-3}$ \\
{\textbf{Distil}$^{\mathbf{1}}$} & $1.69 \times 10^{-4}$ & $3.50 \times 10^{-4}$ & $3.13 \times 10^{-5}$ \\
{\textbf{Distil}$^{\mathbf{1}}_{\mathbf{x,t}}$} & $1.69 \times 10^{-4}$ & $3.50 \times 10^{-4}$ & $1.53 \times 10^{-5}$ \\
{\textbf{Distil}$^{\mathbf{0,1}}$} & $1.69 \times 10^{-4}$ & $3.51 \times 10^{-4}$ & $5.07 \times 10^{-5}$ \\
{\textbf{Distil}$^{\mathbf{0,1}}_{\mathbf{x,t}}$} & $1.68 \times 10^{-4}$ & $3.50 \times 10^{-4}$ & $1.36 \times 10^{-5}$ \

# Normalised MSE

In [27]:
tests = [f"AdvectionReactionDiffusion/{s}/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/intra_test_indices.pth" for s in ["Heat", "AllenCahn", "ConvectionSlow"]]

full_datasets = [f"AdvectionReactionDiffusion/{s}/data/NeumannBC/rectangle/VaryIC_VaryD/5sources_ConstTimeStep_0-9/rep0/full_datasets.pth" for s in ["Heat", "AllenCahn", "ConvectionSlow"]]

stats_dirs = [f"AdvectionReactionDiffusion/{s}" for s in ["Heat", "AllenCahn", "ConvectionSlow"]]

methods = ["distil_Derivative_x+Derivative_t", "distil_Derivative_x+Derivative_t","distil_Derivative_x+Derivative_t"]

systems = ["D", "RD", "AD"]

In [28]:
nmse = {}
for sys, method, test, full_dataset, stats_dir in zip(systems, methods, tests, full_datasets, stats_dirs):

    with open(f"{stats_dir}/intra_test_stats.pkl", "rb") as f:
        intra_stats = pickle.load(f)

    data_dict = load_ds(dataset_path=full_dataset, indices_path=test, incremental_dim="Time")

    nmse[sys] = {}
    for i in [0, 1, 2, 3, 4]:
        ds = data_dict[f"IC{i}"]["all_tasks"]
        ds_no_ic = data_dict[f"IC{i}"]["all_tasks_no_ic"]

        n_points = len(ds)
        n_points_no_ic = len(ds_no_ic)

        u = ds.tensors[U]
        dut = ds_no_ic.tensors[DU][:, 2]
        dux = ds_no_ic.tensors[DU][:, 0]
        duy = ds_no_ic.tensors[DU][:, 1]

        del ds
        del ds_no_ic

        u_var = torch.var(u, unbiased=False)
        du_var = torch.var(dux+duy+dut, unbiased=False)

        u_sum = torch.sum(u**2)
        du_sum = torch.sum(dux**2 + duy**2 + dut**2)

        mse_u = intra_stats[method]["losses"]["all_tasks"][f"IC{i}"]["model0"]["task0"]["out"]
        mse_du = intra_stats[method]["losses"]["all_tasks"][f"IC{i}"]["model0"]["task0"]["der"] * 3

        e_u = mse_u * n_points
        e_du = mse_du * n_points_no_ic

        nmse[sys][f"IC{i}"] = {
            "nmse": {
                "u": e_u / u_sum,
                "du": e_du / du_sum
            },
            "nmse_var": {
                "u": mse_u / u_var,
                "du": mse_du / du_var
            }
        }
    nmse[sys]["avg_nmse"] = {
        "u": np.mean([nmse[sys][f"IC{i}"]["nmse"]["u"] for i in [0, 1, 2, 3, 4]]),
        "du": np.mean([nmse[sys][f"IC{i}"]["nmse"]["du"] for i in [0, 1, 2, 3, 4]])
    }
    nmse[sys]["avg_nmse_var"] = {
        "u": np.mean([nmse[sys][f"IC{i}"]["nmse_var"]["u"] for i in [0, 1, 2, 3, 4]]),
        "du": np.mean([nmse[sys][f"IC{i}"]["nmse_var"]["du"] for i in [0, 1, 2, 3, 4]])
    }
    del data_dict

In [29]:
nmse.keys()

dict_keys(['D', 'RD', 'AD'])

In [30]:
for caption, color in [("Internal test", "red")]:
    cols = [
        [nmse[sys]["avg_nmse"]["u"] for sys in systems],
        [nmse[sys]["avg_nmse"]["du"] for sys in systems],
        [nmse[sys]["avg_nmse_var"]["u"] for sys in systems],
        [nmse[sys]["avg_nmse_var"]["du"] for sys in systems]
    ]

    col_ids = [
        r"$NMSE_{norm}[u^{\theta^*}, u]$",
        r"$NMSE_{norm}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$",
        r"$NMSE_{var}[u^{\theta^*}, u]$",
        r"$NMSE_{var}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$"
    ]
    
    row_ids = [
        r"Heat Diffusion",
        r"Allen-Cahn",
        r"Convection"
    ]

    table = latex_table(
        columns=cols, 
        col_ids=col_ids, 
        row_ids=row_ids, 
        diag_ids=True,
        colorss=[
            get_colors(cols[0], COLOR_SCALES[color], "max"), 
            get_colors(cols[1], COLOR_SCALES[color], "max"), 
            get_colors(cols[2], COLOR_SCALES[color], "max"),
            get_colors(cols[3], COLOR_SCALES[color], "max")
        ],
        col_id_color=COL_ID_COLOR[color], 
        row_id_color=ROW_ID_COLOR[color],
        caption=caption,
        ub=False
    ) +"\n"
    with open(f"{stats_dir}/nmse_table.txt", "w") as file:
        file.write(table)
    print(table)

\begin{table}[H]
\centering
\caption{Internal test}
\label{}
\begin{tabular}{lccccc}
\toprule
 & \color[HTML]{9C3535}\dcol{$NMSE_{norm}[u^{\theta^*}, u]$} & \color[HTML]{9C3535}\dcol{$NMSE_{norm}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$} & \color[HTML]{9C3535}\dcol{$NMSE_{var}[u^{\theta^*}, u]$} & \color[HTML]{9C3535}\dcol{$NMSE_{var}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$} \\
\midrule
\color[HTML]{BE4141}\drow{Heat Diffusion} & $3.92 \times 10^{-5}$ & $1.20 \times 10^{-3}$ & \cellcolor[HTML]{F8ECEC}$1.44 \times 10^{-4}$ & $1.54 \times 10^{-3}$ \\
\color[HTML]{BE4141}\drow{Allen-Cahn} & \cellcolor[HTML]{F8ECEC}$4.05 \times 10^{-5}$ & \cellcolor[HTML]{F8ECEC}$1.35 \times 10^{-3}$ & $7.26 \times 10^{-5}$ & \cellcolor[HTML]{F8ECEC}$1.55 \times 10^{-3}$ \\
\color[HTML]{BE4141}\drow{Convection} & \cellcolor[HTML]{EDCACA}$3.93 \times 10^{-4}$ & \cellcolor[HTML]{EDCACA}$8.11 \times 10^{-3}$ & \cellcolor[HTML]{EDCACA}$9.39 \times 10^{-4}$ & \cellcolor[HTML]{EDCACA}$7.07 \times 10^{-3}$

In [25]:
for caption, color in [("Internal test", "red")]:
    cols = [
        [nmse[sys]["avg_nmse"]["u"] for sys in systems],
        [nmse[sys]["avg_nmse_var"]["u"] for sys in systems]
    ]

    col_ids = [
        r"$NMSE_{norm}[u^{\theta^*}, u]$",
        r"$NMSE_{var}[u^{\theta^*}, u]$"
    ]
    
    row_ids = [
        r"Heat Diffusion",
        r"Allen-Cahn",
        r"Convection"
    ]

    table = latex_table(
        columns=cols, 
        col_ids=col_ids, 
        row_ids=row_ids, 
        diag_ids=False,
        colorss=[
            get_colors(cols[0], COLOR_SCALES[color], "max"), 
            get_colors(cols[1], COLOR_SCALES[color], "max"),
        ],
        col_id_color=COL_ID_COLOR[color], 
        row_id_color=ROW_ID_COLOR[color],
        caption=caption,
        ub=False
    ) +"\n"
    with open(f"{stats_dir}/nmse_table_u.txt", "w") as file:
        file.write(table)
    print(table)

\begin{table}[H]
\centering
\caption{Internal test}
\label{}
\begin{tabular}{lccccc}
\toprule
 & \color[HTML]{9C3535}{$NMSE_{norm}[u^{\theta^*}, u]$} & \color[HTML]{9C3535}{$NMSE_{var}[u^{\theta^*}, u]$} \\
\midrule
\color[HTML]{BE4141}{Heat Diffusion} & \cellcolor[HTML]{F8ECEC}$3.92 \times 10^{-5}$ & \cellcolor[HTML]{F8ECEC}$1.44 \times 10^{-4}$ \\
\color[HTML]{BE4141}{Allen-Cahn} & $3.49 \times 10^{-5}$ & $5.41 \times 10^{-5}$ \\
\color[HTML]{BE4141}{Convection} & \cellcolor[HTML]{EDCACA}$3.93 \times 10^{-4}$ & \cellcolor[HTML]{EDCACA}$9.39 \times 10^{-4}$ \\
\bottomrule
\end{tabular}
\end{table}




In [31]:
for caption, color in [("Internal test", "red")]:
    cols = [
        [nmse[sys]["avg_nmse"]["du"] for sys in systems],
        [nmse[sys]["avg_nmse_var"]["du"] for sys in systems]
    ]

    col_ids = [
        r"$NMSE_{norm}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$",
        r"$NMSE_{var}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$"
    ]
    
    row_ids = [
        r"Heat Diffusion",
        r"Allen-Cahn",
        r"Convection"
    ]

    table = latex_table(
        columns=cols, 
        col_ids=col_ids, 
        row_ids=row_ids, 
        diag_ids=False,
        colorss=[
            get_colors(cols[0], COLOR_SCALES[color], "max"), 
            get_colors(cols[1], COLOR_SCALES[color], "max")
        ],
        col_id_color=COL_ID_COLOR[color], 
        row_id_color=ROW_ID_COLOR[color],
        caption=caption,
        ub=False
    ) +"\n"
    with open(f"{stats_dir}/nmse_table_du.txt", "w") as file:
        file.write(table)
    print(table)

\begin{table}[H]
\centering
\caption{Internal test}
\label{}
\begin{tabular}{lccccc}
\toprule
 & \color[HTML]{9C3535}{$NMSE_{norm}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$} & \color[HTML]{9C3535}{$NMSE_{var}[\nabla_{x,t} u^{\theta^*}, \nabla_{x,t} u]$} \\
\midrule
\color[HTML]{BE4141}{Heat Diffusion} & $1.20 \times 10^{-3}$ & $1.54 \times 10^{-3}$ \\
\color[HTML]{BE4141}{Allen-Cahn} & \cellcolor[HTML]{F8ECEC}$1.35 \times 10^{-3}$ & \cellcolor[HTML]{F8ECEC}$1.55 \times 10^{-3}$ \\
\color[HTML]{BE4141}{Convection} & \cellcolor[HTML]{EDCACA}$8.11 \times 10^{-3}$ & \cellcolor[HTML]{EDCACA}$7.07 \times 10^{-3}$ \\
\bottomrule
\end{tabular}
\end{table}


